# CREATE THE VALIDATION DATASET

In [1]:
from dl_client import DatalakeClient
import pandas as pd
import numpy as np
import os

client = DatalakeClient()

## UTILS

In [2]:
def split_dataset_in_train_val(dataset, percentage):

    # Count visits per RID
    visits_count = dataset.groupby('ID').size()
    
    # Create a dictionary that groups RIDs by number of visits
    sub_per_visits = {}
    for rid, count in visits_count.items():
        if count not in sub_per_visits:
            sub_per_visits[count] = []
        sub_per_visits[count].append(rid)

    train_id = []
    for count in sub_per_visits:
        sublist_id = np.random.choice(sub_per_visits[count], size=int(percentage*len(sub_per_visits[count])), replace=False).tolist()
        train_id += sublist_id

    df_train = dataset[dataset['ID'].isin(train_id)].copy()
    df_val = dataset[~dataset['ID'].isin(train_id)].copy()

    # Get from the test dataset a dataset to predict 
    df_to_pred = df_val.groupby('ID').tail(1).copy() # Get last visit of every patient

    list_index = df_to_pred.index.tolist()
    df_pers = df_val[~df_val.index.isin(list_index)].copy() # Evaluation dataset

    # Set multi index
    df_train = df_train.set_index(['ID', 'TIME'])
    df_pers = df_pers.set_index(['ID', 'TIME'])
    df_to_pred = df_to_pred.set_index(['ID', 'TIME'])

    return df_train, df_val, df_pers, df_to_pred

In [3]:
def get_dataset_custom_metadata(metadata):
    client = DatalakeClient()

    search = client.search_files(
        query=metadata
    )
    
    return search['files'][0]['custom']

def get_dataset_predictors(level, file_code):
    metadata = {
        'custom.level' : level,
        'custom.file_code': file_code
    }

    custom_metadata = get_dataset_custom_metadata(metadata)

    return custom_metadata['predittori']
    
def get_dataset_cofactors(level, file_code):
    metadata = {
        'custom.level' : level,
        'custom.file_code': file_code
    }

    custom_metadata = get_dataset_custom_metadata(metadata)

    return custom_metadata['cofattori']

In [4]:
def remove_unnecessary_values_from_list(list_values, unnecessary_values):
    return [value for value in list_values if value not in unnecessary_values]

In [5]:
def prepare_dataset(dataset, predictors, cofactors, exclude = []):
    """
    Prepare the dataset for the model.
    
    This function does the following:
    - Rename columns if necessary (RID→ID, AGE→TIME)
    - Select all required columns (both predictors and cofactors)
    - Make id as a string
    - Remove rows with NaN values in the TIME column
    - Count visits per ID
    - Get IDs with more than one visit
    - Filter dataset to include only patients with multiple visits
    
    Parameters:
    -----------
    dataset : pandas.DataFrame
        The dataset to prepare
    
    Returns:
    --------
    pandas.DataFrame
        A copy of the input dataframe with the required columns
    """
    # Rename columns if necessary (RID→ID, AGE→TIME)
    dataset = dataset.rename(columns={'RID': 'ID'})
    dataset = dataset.rename(columns={'AGE': 'TIME'})

    # Select all required columns (both predictors and cofactors)
    # Predictors
    predictor_columns = predictors + ['ID', 'TIME']
    # Cofactors
    cofactor_columns = cofactors

    if exclude:
        predictor_columns = remove_unnecessary_values_from_list(predictor_columns, exclude)
        cofactor_columns = remove_unnecessary_values_from_list(cofactor_columns, exclude)

    # Combine all columns to keep
    columns_to_keep = list(set(predictor_columns + cofactor_columns))
    
    # Check if the columns exist in the dataset
    available_columns = [col for col in columns_to_keep if col in dataset.columns]
    dataset = dataset[available_columns]
    
    # Make id as a string
    dataset['ID'] = dataset['ID'].astype('str') 
    
    # Remove rows with NaN values in the TIME column
    dataset = dataset.dropna(subset=['TIME'])

    return dataset

In [6]:
def load_dataset_to_datalake(metadata, data, filename, prefix):
    client = DatalakeClient()

    try:
        search = client.upload_dataframe(
            df=data,
            object_name=filename,
            prefix=prefix,
            metadata=metadata
        )

        return True
    except Exception as e:
        print('Upload on datalake failed: ', e)
        return False

## VARIABLES

In [7]:
file_code = 'ADNIMERGE'
level = 'cleaned_03'
prefix = 'validation'
new_filename = 'dataset_for_validation'
subset_percentage = 0.8
columns_to_exclude=['AGE', 'DX/CN', 'DX/Dementia', 'DX/MCI', "ICV%ICV"]

## READ DATA

In [8]:
search = client.query_files(
    query={
        'custom.level' : level,
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

## PREPARE THE VALIDATION DATASET

### GET PREDICTORS AND COFACTOR FROM DATALAKE

In [9]:
predictors = get_dataset_predictors(level, file_code)
cofactors = get_dataset_cofactors(level, file_code)

print(predictors)
print(cofactors)

['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV', 'ICV%ICV']
['EDUCAT', 'APOE_4', 'GENDER/female', 'GENDER/male', 'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed', 'ETHNICITY/latino', 'ETHNICITY/not_latino', 'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american', 'RACE/White']


### EXLUDE SOME VARIABLES AND RENAME THE COLUMNS

In [10]:
dataset = prepare_dataset(df_new, predictors, cofactors, exclude=columns_to_exclude)

# Convert boolean columns to integer
for col in cofactors:
    if col in dataset.columns and dataset[col].dtype == 'bool':
        dataset[col] = dataset[col].astype(int)

### GET SUBSET

In [11]:
df_train, df_val, _, _ = split_dataset_in_train_val(dataset, 0.8)
display(df_train)

Entorhinal%ICV  CDRSB  MMSE  RAVLT_immediate  MARRY/widowed  \
ID   TIME                                                                
2    74.3        0.210464    0.0  28.0             44.0              0   
     74.7             NaN    0.0  28.0             40.0              0   
     77.2             NaN    0.0  29.0             34.0              0   
     79.3             NaN    0.0  28.0             37.0              0   
     80.3             NaN    0.0  23.0             42.0              0   
...                   ...    ...   ...              ...            ...   
7083 76.0             NaN    6.5  25.0              NaN              0   
7079 86.7        0.241376    0.5  27.0             34.0              1   
     87.6             NaN    1.0  27.0             35.0              1   
7088 70.0        0.271960    1.5  27.0             24.0              0   
     70.9             NaN    2.0  25.0             24.0              0   

           GENDER/male  ETHNICITY/not_latino  Ventricles%ICV  \
ID   TIME                                                      
2    74.3            1                     1        5.957343   
     74.7            1                     1             NaN   
     77.2            1                     1             NaN   
     79.3            1                     1             NaN   
     80.3            1                     1             NaN   
...                ...                   ...             ...   
7083 76.0            1                     1             NaN   
7079 86.7            0                     1        2.378043   
     87.6            0                     1             NaN   
7088 70.0            1                     1        3.295828   
     70.9            1                     1             NaN   

           RACE/Native_american  MARRY/divorced  ...  ADAS11  RACE/Mixed  \
ID   TIME                                        ...                       
2    74.3                     0               0  ...   10.67           0   
     74.7                     0               0  ...   10.67           0   
     77.2                     0               0  ...   12.00           0   
     79.3                     0               0  ...   14.00           0   
     80.3                     0               0  ...   12.00           0   
...                         ...             ...  ...     ...         ...   
7083 76.0                     0               0  ...   14.67           0   
7079 86.7                     0               0  ...    9.00           0   
     87.6                     0               0  ...    7.00           0   
7088 70.0                     0               0  ...   15.67           0   
     70.9                     0               0  ...   16.67           0   

           GENDER/female  ADAS13  EDUCAT  RACE/Asian  APOE_4  Hippocampus%ICV  \
ID   TIME                                                                       
2    74.3              0   18.67      16           0     0.0         0.420022   
     74.7              0   19.67      16           0     0.0              NaN   
     77.2              0   20.00      16           0     0.0              NaN   
     79.3              0   23.00      16           0     0.0              NaN   
     80.3              0   21.00      16           0     0.0              NaN   
...                  ...     ...     ...         ...     ...              ...   
7083 76.0              0   24.67      17           0     NaN              NaN   
7079 86.7              1   16.00      18           0     NaN         0.511938   
     87.6              1   16.00      18           0     NaN              NaN   
7088 70.0              0   24.67      16           0     NaN         0.391396   
     70.9              0   25.67      16           0     NaN              NaN   

           ETHNICITY/latino  MARRY/single  
ID   TIME                                  
2    74.3                 0             0  
     74.7                 0             

In [12]:
df_for_validation = df_train.reset_index()
if 'index' in df_for_validation.columns:
    df_for_validation = df_for_validation.drop('index', axis=1)

display(df_for_validation)

,ID,TIME,Entorhinal%ICV,CDRSB,MMSE,RAVLT_immediate,MARRY/widowed,GENDER/male,ETHNICITY/not_latino,Ventricles%ICV,...,ADAS11,RACE/Mixed,GENDER/female,ADAS13,EDUCAT,RACE/Asian,APOE_4,Hippocampus%ICV,ETHNICITY/latino,MARRY/single
0,2,74.3,0.210464,0.0,28.0,44.0,0,1,1,5.957343,...,10.67,0,0,18.67,16,0,0.0,0.420022,0,0
1,2,74.7,NaN,0.0,28.0,40.0,0,1,1,NaN,...,10.67,0,0,19.67,16,0,0.0,NaN,0,0
2,2,77.2,NaN,0.0,29.0,34.0,0,1,1,NaN,...,12.00,0,0,20.00,16,0,0.0,NaN,0,0
3,2,79.3,NaN,0.0,28.0,37.0,0,1,1,NaN,...,14.00,0,0,23.00,16,0,0.0,NaN,0,0
4,2,80.3,NaN,0.0,23.0,42.0,0,1,1,NaN,...,12.00,0,0,21.00,16,0,0.0,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8818,7083,76.0,NaN,6.5,25.0,NaN,0,1,1,NaN,...,14.67,0,0,24.67,17,0,NaN,NaN,0,0
8819,7079,86.7,0.241376,0.5,27.0,34.0,1,0,1,2.378043,...,9.00,0,1,16.00,18,0,NaN,0.511938,0,0
8820,7079,87.6,NaN,1.0,27.0,35.0,1,0,1,NaN,...,7.00,0,1,16.00,18,0,NaN,NaN,0,0
8821,7088,70.0,0.271960,1.5,27.0,24.0,0,1,1,3.295828,...,15.67,0,0,24.67,16,0,NaN,0.391396,0,0


In [13]:
filename_source = new_filename + '.csv'
metadata_for_validation = {
    'file_code': file_code + '_for_validation',
    'level': level,
}

load_dataset_to_datalake(metadata_for_validation, df_for_validation, filename_source, prefix)

True